# Zero-Shot Pre-Labeling for Negative and Positive Reviews

This notebook uses a zero-shot classification model to pre-label PickMe reviews.

We create two separate labeled datasets:

1. Negative reviews → complaint categories
2. Positive reviews → satisfaction categories

These pre-labeled datasets will later be manually reviewed and corrected.

In [1]:
import pandas as pd
from pathlib import Path
from transformers import pipeline
from tqdm import tqdm
import torch

In [2]:
RAW_PATH = Path("../data/raw/pickme_reviews_with_sentiment.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Load English reviews and split by sentiment

In [3]:
df = pd.read_csv(RAW_PATH)

# Keep English reviews
df_en = df[df["language"].astype(str).str.strip().str.lower() == "english"].copy()

# Clean review text
df_en["review_text"] = df_en["review_text"].astype(str).str.strip()

# Remove empty reviews
df_en = df_en[df_en["review_text"].str.len() > 0].reset_index(drop=True)

# Clean sentiment column
df_en["sentiment_clean"] = df_en["sentiment"].astype(str).str.strip().str.upper()

print(df_en["sentiment_clean"].value_counts())

sentiment_clean
NEGATIVE    847
POSITIVE    593
Name: count, dtype: int64


In [6]:
#split into positive and negative reviews
df_neg = df_en[df_en["sentiment"] == "Negative"].copy().reset_index(drop=True)
df_pos = df_en[df_en["sentiment"] == "Positive"].copy().reset_index(drop=True)

print(f"Negative reviews: {len(df_neg)}")
print(f"Positive reviews: {len(df_pos)}")

Negative reviews: 847
Positive reviews: 593


Define category labels

In [4]:
candidate_labels = [
    "App bugs and technical issues",
    "Pricing, surge, and bidding",
    "Food delivery delays and missing items",
    "Driver behavior and safety",
    "Customer support and refunds",
    "Privacy and security concerns"
]

Initialize the zero-shot pipeline

In [5]:
print("Loading model...")
classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=0
)

Loading model...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]